In [1]:
import pandas as pd
import numpy as np
import pickle
import copy
import re

In [ ]:
data = pd.read_csv(r'AC_cast硬度数据集_483.csv',index_col = 0)
# data = pd.read_csv(r'D:\work\LLM\硬度预测\PTR论文数据\Ac_cast\数据\AC_cast硬度数据集_483.csv',index_col = 0)
data.index = [i for i in range(len(data))]
print(data)
print(data.columns)
cloumn_list = []
for i in range(len(data.columns)):
    cloumn_list.append([i, data.columns[i]])
print(cloumn_list)

                                          Composition       VEC  \
0                    Al0.017 Cu0.328 Mn0.328 Ni0.328   9.225774   
1    Al0.020 Co0.196 Cr0.196 Fe0.196 Mn0.196 Ni0.196   7.900000   
2            Al0.024 Co0.244 Cr0.244 Fe0.244 Ni0.244   8.124000   
3                    Al0.036 Cu0.321 Mn0.321 Ni0.321   9.105105   
4    Al0.038 Co0.192 Cr0.192 Fe0.192 Mn0.192 Ni0.192   7.809619   
..                                                ...       ...   
478                   Nb0.250 Ti0.250 V0.250 Zr0.250   4.500000   
479                          Nb0.333 Ta0.333 Ti0.333   4.666667   
480                          Nb0.333 Ti0.333 Zr0.333   4.333333   
481                  Nb0.375 Ta0.250 Ti0.250 Zr0.125   4.625000   
482                          Ta0.333 Ti0.333 Zr0.333   4.333333   

     Electronegativity_Difference  Atomic_Radius_Diff  Mixing_Enthalpy  \
0                        0.167533            3.617080        -0.934897   
1                        0.138701            3.

In [3]:
# 定义设计的几组成分（原子百分比）
alloy_information = [
    {'Al': 20, 'Nb': 28, 'Ti': 20, 'V': 4, 'Cr': 20, 'Mo': 8, 'HV': 593.8},
    {'Al': 14, 'Nb': 22, 'Ti': 30, 'V': 2, 'Cr': 20, 'Mo': 12, 'HV': 518.5},
    {'Al': 8, 'Nb': 22, 'Ti': 34, 'V': 4, 'Cr': 20, 'Mo': 12, 'HV': 507.4},
]

In [4]:
# 检查这几组成分是否在数据集中

In [5]:
# 设置容忍度（2%的绝对差异）
tolerance = 0.02

# 获取所有元素列
element_columns = ['Li', 'Mg', 'Al', 'Si', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 
                   'Ni', 'Co', 'Cu', 'Zn', 'Zr', 'Nb', 'Mo', 'Sn', 'Hf', 'Ta', 'W']

print("检查目标成分是否在数据集中...")
print("=" * 80)

# 检查每个目标成分
for target_idx, target_comp in enumerate(alloy_information):
    print(f"\n检查目标合金 {target_idx+1}: {target_comp}")
    
    # 提取目标成分中的元素（排除HV）
    target_hv = target_comp.get('HV', None)
    target_elements = {k: v/100.0 for k, v in target_comp.items() if k != 'HV'}
    
    found = False
    
    # 遍历数据集中的每一行
    for data_idx in range(len(data)):
        # 步骤1: 获取当前数据行的非零元素
        data_elements = {}
        for element in element_columns:
            value = data.loc[data_idx, element]
            if value > 0.001:  # 认为大于0.1%的元素是存在的
                data_elements[element] = value
        
        # 步骤2: 比较元素体系是否完全一致
        if set(target_elements.keys()) != set(data_elements.keys()):
            continue  # 元素体系不一致，跳过
        
        # 步骤3: 比较每个元素的含量
        match = True
        for element, target_value in target_elements.items():
            data_value = data_elements.get(element, 0)
            if abs(data_value - target_value) > tolerance:
                match = False
                break
        
        # 如果匹配成功
        if match:
            found = True
            hv_value = data.loc[data_idx, 'HV']
            print(f"  ✅ 找到了！")
            print(f"     数据集索引: {data_idx}")
            print(f"     数据集成分: {data.loc[data_idx, 'Composition']}")
            print(f"     数据集HV: {hv_value}")
            print(f"     目标HV: {target_hv}")
            
            # 显示元素对比
            print("     元素含量对比:")
            for element, target_value in target_elements.items():
                data_value = data_elements[element]
                diff = abs(data_value - target_value) * 100  # 转换为百分比
                print(f"       {element}: 目标={target_value*100:.1f}%, " +
                      f"实际={data_value*100:.1f}%, 差异={diff:.1f}%")
            
            # 找到后就可以跳出循环了
            break
    
    if not found:
        print(f"  ❌ 未在数据集中找到这个合金")

检查目标成分是否在数据集中...

检查目标合金 1: {'Al': 20, 'Nb': 28, 'Ti': 20, 'V': 4, 'Cr': 20, 'Mo': 8, 'HV': 593.8}
  ❌ 未在数据集中找到这个合金

检查目标合金 2: {'Al': 14, 'Nb': 22, 'Ti': 30, 'V': 2, 'Cr': 20, 'Mo': 12, 'HV': 518.5}
  ❌ 未在数据集中找到这个合金

检查目标合金 3: {'Al': 8, 'Nb': 22, 'Ti': 34, 'V': 4, 'Cr': 20, 'Mo': 12, 'HV': 507.4}
  ❌ 未在数据集中找到这个合金


In [6]:
# 生成个元素的嵌入向量并且保存。

In [ ]:
# save_element_embeddings.py
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import json
import os

# 定义所有可能的元素
ALL_ELEMENTS = [
    'Li', 'Mg', 'Al', 'Si', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe',
    'Ni', 'Co', 'Cu', 'Zn', 'Zr', 'Nb', 'Mo', 'Sn', 'Hf', 'Ta', 'W'
]

def extract_and_save_element_embeddings():
    """提取并保存所有元素的嵌入向量"""
    # 1. 初始化模型
    model_path = r"model_results_save\final_model"
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print(f"加载模型，使用设备: {device}")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModel.from_pretrained(model_path).to(device)
    model.eval()
    
    # 2. 提取嵌入向量
    embeddings = {}
    for element in ALL_ELEMENTS:
        # Tokenize
        inputs = tokenizer(
            element,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=256
        ).to(device)
        
        # 获取嵌入
        with torch.no_grad():
            outputs = model(**inputs)
        
        # 使用[CLS] token的嵌入
        embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy().flatten()
        embeddings[element] = embedding.tolist()  # 转换为列表便于JSON保存
    
    # 3. 保存到文件
    save_dir = './element_embeddings'
    os.makedirs(save_dir, exist_ok=True)
    
    # 保存嵌入向量
    with open(os.path.join(save_dir, 'embeddings.json'), 'w') as f:
        json.dump(embeddings, f)
    
    print(f"✓ 已保存 {len(embeddings)} 个元素的嵌入向量到 {save_dir}")
    return save_dir

if __name__ == "__main__":
    extract_and_save_element_embeddings()

加载模型，使用设备: cuda
✓ 已保存 21 个元素的嵌入向量到 ./element_embeddings


In [9]:
# 生成设计合金的嵌入矩阵

In [10]:
# generate_alloy_matrix.py
import numpy as np
import json

def load_element_embeddings():
    """加载保存的元素嵌入"""
    with open('./element_embeddings/embeddings.json', 'r') as f:
        embeddings = json.load(f)
    # 将列表转回numpy数组
    return {k: np.array(v) for k, v in embeddings.items()}

def normalize_composition(composition_dict):
    """
    归一化合金成分，移除HV键
    
    参数:
    composition_dict: 包含元素含量和HV的字典
    
    返回:
    normalized: 归一化后的成分字典
    hv_value: HV值
    """
    # 复制字典以避免修改原数据
    comp_copy = composition_dict.copy()
    
    # 提取HV值
    hv_value = comp_copy.pop('HV', None)
    
    # 计算总含量
    total = sum(v for k, v in comp_copy.items())
    
    # 归一化到总和为1
    normalized = {k: v/total for k, v in comp_copy.items()}
    
    return normalized, hv_value

def generate_single_alloy_matrix(composition_dict, embeddings):
    """
    为单个合金生成嵌入矩阵
    
    参数:
    composition_dict: 归一化后的成分字典
    embeddings: 元素嵌入字典
    
    返回:
    matrix: 合金的嵌入矩阵 (n_elements, embedding_dim)
    """
    element_names = list(embeddings.keys())
    embedding_dim = len(embeddings[element_names[0]])
    
    # 创建矩阵 (n_elements, embedding_dim)
    matrix = np.zeros((len(element_names), embedding_dim))
    
    for i, elem in enumerate(element_names):
        if elem in composition_dict:
            # 元素存在，使用归一化含量加权
            weight = composition_dict[elem]
            matrix[i, :] = embeddings[elem] * weight
    
    return matrix

def generate_all_alloy_matrices(alloy_list):
    """
    为多个合金生成嵌入矩阵并合并
    
    参数:
    alloy_list: 合金字典列表
    
    返回:
    combined_matrix: 合并后的三维矩阵 (n_alloys, n_elements, embedding_dim)
    hv_values: HV值列表
    normalized_compositions: 归一化后的成分列表
    """
    # 1. 加载元素嵌入
    embeddings = load_element_embeddings()
    element_names = list(embeddings.keys())
    embedding_dim = len(embeddings[element_names[0]])
    
    n_alloys = len(alloy_list)
    n_elements = len(element_names)
    
    # 2. 预分配三维矩阵 (n_alloys, n_elements, embedding_dim)
    combined_matrix = np.zeros((n_alloys, n_elements, embedding_dim))
    hv_values = []
    normalized_compositions = []
    
    # 3. 处理每个合金
    for idx, alloy in enumerate(alloy_list):
        print(f"\n处理合金 {idx+1}:")
        print("-" * 30)
        print(f"原始成分: {alloy}")
        
        # 归一化成分
        normalized, hv_value = normalize_composition(alloy)
        normalized_compositions.append(normalized)
        hv_values.append(hv_value)
        
        # 显示归一化结果
        print("归一化后的成分:")
        for elem, percent in normalized.items():
            print(f"  {elem}: {percent:.4f} ({percent*100:.2f}%)")
        
        # 生成单个合金矩阵
        matrix = generate_single_alloy_matrix(normalized, embeddings)
        combined_matrix[idx, :, :] = matrix
        
        print(f"生成的矩阵形状: {matrix.shape}")
        print(f"HV值: {hv_value}")
    
    return combined_matrix, hv_values, normalized_compositions

# 示例使用
if __name__ == "__main__":
    
    # 生成所有合金的嵌入矩阵
    combined_matrix, hv_values, normalized_compositions = generate_all_alloy_matrices(alloy_information)
    
    # 打印汇总信息
    print("\n" + "="*60)
    print("汇总信息:")
    print("="*60)
    print(f"合并后的矩阵形状: {combined_matrix.shape}")
    print(f"  第一个维度: {combined_matrix.shape[0]} 个合金")
    print(f"  第二个维度: {combined_matrix.shape[1]} 个元素")
    print(f"  第三个维度: {combined_matrix.shape[2]} 维嵌入")
    
    print(f"\nHV值列表: {hv_values}")


处理合金 1:
------------------------------
原始成分: {'Al': 20, 'Nb': 28, 'Ti': 20, 'V': 4, 'Cr': 20, 'Mo': 8, 'HV': 593.8}
归一化后的成分:
  Al: 0.2000 (20.00%)
  Nb: 0.2800 (28.00%)
  Ti: 0.2000 (20.00%)
  V: 0.0400 (4.00%)
  Cr: 0.2000 (20.00%)
  Mo: 0.0800 (8.00%)
生成的矩阵形状: (21, 768)
HV值: 593.8

处理合金 2:
------------------------------
原始成分: {'Al': 14, 'Nb': 22, 'Ti': 30, 'V': 2, 'Cr': 20, 'Mo': 12, 'HV': 518.5}
归一化后的成分:
  Al: 0.1400 (14.00%)
  Nb: 0.2200 (22.00%)
  Ti: 0.3000 (30.00%)
  V: 0.0200 (2.00%)
  Cr: 0.2000 (20.00%)
  Mo: 0.1200 (12.00%)
生成的矩阵形状: (21, 768)
HV值: 518.5

处理合金 3:
------------------------------
原始成分: {'Al': 8, 'Nb': 22, 'Ti': 34, 'V': 4, 'Cr': 20, 'Mo': 12, 'HV': 507.4}
归一化后的成分:
  Al: 0.0800 (8.00%)
  Nb: 0.2200 (22.00%)
  Ti: 0.3400 (34.00%)
  V: 0.0400 (4.00%)
  Cr: 0.2000 (20.00%)
  Mo: 0.1200 (12.00%)
生成的矩阵形状: (21, 768)
HV值: 507.4

汇总信息:
合并后的矩阵形状: (3, 21, 768)
  第一个维度: 3 个合金
  第二个维度: 21 个元素
  第三个维度: 768 维嵌入

HV值列表: [593.8, 518.5, 507.4]


In [11]:
print(combined_matrix, hv_values, normalized_compositions)

[[[ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.20521107  0.09945132 -0.06108301 ... -0.30499551 -0.00799374
   -0.17302436]
  ...
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]]

 [[ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.14364775  0.06961592 -0.0427581  ... -0.21349686 -0.00559562
   -0.12111705]
  ...
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]
  [ 0.          0.          0.         ...  0.          0.
    0.        ]]

 [[ 0.          0.          0.         ...  0.          0.
    0.   

In [12]:
# 调用模型进行预测

In [13]:
print(combined_matrix.shape)

(3, 21, 768)


In [14]:
for index in range(len(alloy_information)):
    print(alloy_information[index])

{'Al': 20, 'Nb': 28, 'Ti': 20, 'V': 4, 'Cr': 20, 'Mo': 8, 'HV': 593.8}
{'Al': 14, 'Nb': 22, 'Ti': 30, 'V': 2, 'Cr': 20, 'Mo': 12, 'HV': 518.5}
{'Al': 8, 'Nb': 22, 'Ti': 34, 'V': 4, 'Cr': 20, 'Mo': 12, 'HV': 507.4}


In [15]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

In [ ]:
# -*- coding: utf-8 -*-
"""
最简化版本 - 单个文件完成预测
直接运行此脚本即可
"""
import joblib

import os
os.environ['PYTHONHASHSEED'] = '42'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

# import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model

# GPU内存配置
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f"GPU配置警告: {e}")

# ===========================================================================
# 在这里修改你的配置
# ===========================================================================


save_dir = './alloy_prediction_features'
os.makedirs(save_dir, exist_ok=True)

# 1. 设置模型路径from pathlib import Path
from pathlib import Path

model_name = f"best_model_final_0\\best_model_fold_0.h5"

MODEL_PATH = model_name
# 使用时: str(MODEL_PATH) 转换为字符串
# ===========================================================================

# ===========================================================================

print("="*60)
print(f"{model_name}模型预测")
print("="*60)

# 检查数据形状
# print(f"\n输入数据形状: {combined_matrix.shape}")
assert combined_matrix.shape[1:] == (21, 768), "数据形状必须是 (n_samples, 21, 768)"

# 调整形状以匹配模型输入
X_pred = np.expand_dims(combined_matrix, axis=1)  # (n_samples, 1, 21, 768)
# print(f"调整后形状: {X_pred.shape}")

# 加载模型
# print(f"\n加载模型: {os.path.basename(MODEL_PATH)}")
model = load_model(MODEL_PATH)

# 预测
# print("\n预测中...")
predictions = model.predict(X_pred, verbose=0).reshape(-1)
# 显示结果
print("\n" + "="*60)
print("预测结果:")
print("="*60)
for i, hv in enumerate(predictions):
    print(f"样本 {i+1}: HV = {hv:.4f}", abs(hv-alloy_information[i]["HV"]))

y_true = [alloy_information[i]["HV"] for i in range(len(alloy_information))]
# MAE (平均绝对误差)
mae = mean_absolute_error(y_true, predictions)
# MSE (均方误差)
mse = mean_squared_error(y_true, predictions)
# RMSE (均方根误差)
rmse = np.sqrt(mse)
mape = mean_absolute_percentage_error(y_true, predictions)  # MAPE
r2 = r2_score(y_true, predictions)
print(f"MAE: {mae:.4f}", f"RMSE: {rmse:.4f}", f"MAPE: {mape:.4f}", f"R2: {r2:.4f}")

best_model_final_0\best_model_fold_0.h5模型预测

预测结果:
样本 1: HV = 591.8511 1.9488647460937045
样本 2: HV = 570.5704 52.0704345703125
样本 3: HV = 514.1505 6.750451660156273
MAE: 20.2566 RMSE: 30.3353 MAPE: 0.0390 R2: 0.3753
